<a href="https://colab.research.google.com/github/noddleslee/vlm-federated-lora-pathvqa/blob/main/train_qwen_vlm_pathvqa_lora_top2000_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth

In [ ]:
import torch
from datasets import load_dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig


In [ ]:
# 1. load  vlm model
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
) # tokenizer把文本变为模型要学习的序列，也可以把序列还原成文本

#
rank = 64
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # 1 位置：在vision，language，attetnion，mlp 加载lora模型
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = rank,           # 2 lora 多大: rank
    lora_alpha = rank * 2,   # 3 训练强度和正则，alpha 控制 LoRA 增量 ΔW 在总输出里占多大比重，alpha 越大，学的更快但可能不稳定
    lora_dropout = 0,    # 如果发现过拟合，可以启动dropout
    bias = "none",
    random_state = 3407,  # 4 LoRA 复现和其他高级选项
    use_rslora = False,
    loftq_config = None,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Unsloth: Making `model.base_model.model.model.visual` require gradients


In [ ]:
# 1. load  vlm model
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
) # tokenizer把文本变为模型要学习的序列，也可以把序列还原成文本

# 定义 LoRA 的位置，矩阵的秩和训练的参数。
rank = 64

# 使用 FastVisionModel.get_peft_model 为模型添加 LoRA 适配器，实现PEFT。
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # 在视觉层应用 LoRA 适配器
    finetune_language_layers   = True, # 在语言层应用 LoRA 适配器
    finetune_attention_modules = True, # 在注意力模块应用 LoRA 适配器
    finetune_mlp_modules       = True, # 在 MLP 模块应用 LoRA 适配器
    r = rank,                          # LoRA 矩阵的秩
    lora_alpha = rank * 2,             # LoRA 缩放因子，控制 LoRA 增量更新的权重，alpha越大，学习越快但可能不稳定
    lora_dropout = 0,                  # LoRA 适配器中的 dropout 比率，0表示不使用。若有过拟合可启用。
    bias = "none",                     # 是否为 LoRA 层添加偏置。
    random_state = 3407,               # 设置随机种子，用于保证实验复现性
    use_rslora = False,                # 是否使用 Re-scaled LoRA
    loftq_config = None,               # LoftQ 配置，用于量化 LoRA 权重
)


==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model.visual` require gradients


In [ ]:
# 3. Data preparation and splitting: replace with medical pathology QA dataset
print("Downloading PathVQA dataset...")
# Load directly from Hugging Face, no permissions required
dataset = load_dataset("flaviagiammarino/path-vqa", split="train")

# [Experiment Tip]: To see the effect of Rank as fast as possible, we don't need to run on the full data (full run takes too long).
# We intentionally take only the first 2000 items to create an "information bottleneck", which can greatly amplify the performance differences among different Ranks!
dataset = dataset.select(range(2000))

# Split 10% as validation set to observe eval_loss
dataset = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

def convert_to_conversation(sample):
    # PathVQA dataset provides specific questions and answers
    instruction = sample["question"]

    return {
        "messages": [
            { "role": "user",
              "content" : [
                {"type" : "text",  "text"  : instruction}, # 提取用户输入的文本
                # Note: If the image is black and white or channels are incorrect, convert to RGB to ensure Qwen can process it properly
                {"type" : "image", "image" : sample["image"].convert("RGB")} ] # 提取用户输入的图片 & 统一图片的格式为彩色的
            },
            { "role" : "assistant",
              "content" : [
                {"type" : "text",  "text"  : sample["answer"]} ] # 提assistant生成的答案
            },
        ]
    }

converted_train_dataset = [convert_to_conversation(sample) for sample in train_dataset]
converted_eval_dataset = [convert_to_conversation(sample) for sample in eval_dataset]

print(f"Training set size: {len(converted_train_dataset)}, Validation set size: {len(converted_eval_dataset)}")


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-f2d0e9ef9f022d(…):   0%|          | 0.00/42.8M [00:00<?, ?B/s]

data/train-00001-of-00007-47d8e0220bf6c9(…):   0%|          | 0.00/81.0M [00:00<?, ?B/s]

data/train-00002-of-00007-7fb5037c4c5da7(…):   0%|          | 0.00/104M [00:00<?, ?B/s]

data/train-00003-of-00007-74b9b7b81cc55f(…):   0%|          | 0.00/90.0M [00:00<?, ?B/s]

...

data/test-00002-of-00003-162830843501982(…):   0%|          | 0.00/69.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19654 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6259 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6719 [00:00<?, ? examples/s]

Training set size: 1800, Validation set size: 200


In [ ]:
print("Original train_dataset sample (before conversion):")
display(train_dataset[0])

Original train_dataset sample (before conversion):


{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=792x528>,
 'question': 'what does this image show?',
 'answer': 'oil wrights excellent tropho'}

In [ ]:
print("Converted train_dataset sample (after conversion):")
display(converted_train_dataset[0])

Converted train_dataset sample (after conversion):


{'messages': [{'role': 'user',
   'content': [{'type': 'text', 'text': 'what does this image show?'},
    {'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=792x528>}]},
  {'role': 'assistant',
   'content': [{'type': 'text', 'text': 'oil wrights excellent tropho'}]}]}

In [ ]:
# 4. Configure and start training
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_train_dataset,
    eval_dataset = converted_eval_dataset, # Pass in evaluation dataset
    args = SFTConfig(
        # 训练规模
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1,
        num_train_epochs = 1,            # Train for 1 full Epoch

        # 学习率
        learning_rate = 2e-4,
        warmup_steps = 5,
        lr_scheduler_type = "linear",

        # 打印日志 & 确定评估方式
        logging_steps = 10,
        per_device_eval_batch_size = 8,   # Set evaluation batch size
        eval_strategy = "steps",      # Evaluate by steps
        eval_steps = 100,          # Evaluate every 100 steps

        # 优化器与正则
        optim = "adamw_8bit",
        weight_decay = 0.001,

        # 模型复现&是否画图
        seed = 3407,
        output_dir = "outputs" + "_" + str(rank),
        report_to = "none",         # Change to "wandb" if you use wandb

        # Required parameters for Vision training:
        remove_unused_columns = False,
        dataset_text_field = "",
        disable_tqdm = True,
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

trainer_stats = trainer.train()

# 5. Save fine-tuned LoRA weights
model.save_pretrained("qwen_lora_1epoch")
tokenizer.save_pretrained("qwen_lora_1epoch")

print(f"Training completed! Total time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes.")

Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,800 | Num Epochs = 1 | Total steps = 225
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 92,442,624 of 2,305,684,288 (4.01% trained)


{'loss': '2.031', 'grad_norm': '8.96', 'learning_rate': '0.0001964', 'epoch': '0.04444'}
{'loss': '1.19', 'grad_norm': '4.62', 'learning_rate': '0.0001873', 'epoch': '0.08889'}
{'loss': '1.125', 'grad_norm': '4.04', 'learning_rate': '0.0001782', 'epoch': '0.1333'}
{'loss': '0.9985', 'grad_norm': '2.966', 'learning_rate': '0.0001691', 'epoch': '0.1778'}
{'loss': '1.04', 'grad_norm': '5.518', 'learning_rate': '0.00016', 'epoch': '0.2222'}
{'loss': '1.172', 'grad_norm': '3.474', 'learning_rate': '0.0001509', 'epoch': '0.2667'}
{'loss': '0.8355', 'grad_norm': '3.292', 'learning_rate': '0.0001418', 'epoch': '0.3111'}
{'loss': '0.8977', 'grad_norm': '2.925', 'learning_rate': '0.0001327', 'epoch': '0.3556'}
{'loss': '0.8098', 'grad_norm': '2.603', 'learning_rate': '0.0001236', 'epoch': '0.4'}
{'loss': '0.8471', 'grad_norm': '3.08', 'learning_rate': '0.0001145', 'epoch': '0.4444'}
{'eval_loss': '0.6978', 'eval_runtime': '57.95', 'eval_samples_per_second': '3.451', 'eval_steps_per_second': '0.4

In [1]:
# !pip -q install plotly

import plotly.express as px
import pandas as pd

train_points = [
    (0.04444, 2.031),
    (0.08889, 1.19),
    (0.1333, 1.125),
    (0.1778, 0.9985),
    (0.2222, 1.04),
    (0.2667, 1.172),
    (0.3111, 0.8355),
    (0.3556, 0.8977),
    (0.4, 0.8098),
    (0.4444, 0.8471),
    (0.4889, 0.6388),
    (0.5333, 0.8647),
    (0.5778, 0.762),
    (0.6222, 0.6605),
    (0.6667, 0.6506),
    (0.7111, 0.7713),
    (0.7556, 0.6143),
    (0.8, 0.5736),
    (0.8444, 0.5674),
    (0.8889, 0.6486),
    (0.9333, 0.583),
    (0.9778, 0.6306),
]

df = pd.DataFrame(train_points, columns=["epoch", "train_loss"])

fig = px.line(
    df,
    x="epoch",
    y="train_loss",
    markers=True,
    hover_data={"epoch":":.4f", "train_loss":":.4f"},
    title="Training Loss vs Epoch (Interactive)"
)
fig.update_traces(mode="lines+markers", hovertemplate="epoch=%{x:.4f}<br>loss=%{y:.4f}<extra></extra>")
fig.show()